In [1]:
batch_size = 8 
data_files = [
    {"label": [0], "topology_filename": "../data/md-2025-03-11/FP11ns.prmtop", "coordinates_filename": "../data/md-2025-03-11/FP11-150ns.nc"},
    {"label": [1], "topology_filename": "../data/md-2025-03-11/FP11ns.prmtop", "coordinates_filename": "../data/md-2025-03-11/FP11-150ns.nc"}
]
selection_string = "name CA"
epochs = 5
test_size = 0.20
random_state = 42
lr=0.0001

# 2025-03-16 - MLMD - GCN - random split

In this notebook, we attempt to predict whether our protein is interacting with the agonist or the antagonist. We do so by using a very simple graph neural network that combines graph convolutions for residue types and atom types. In an RTX2060, it takes around 3 minutes to run 5 epochs.


## Things that we want to know

- What's the region that provides the most informative description? We control this with the `selection_string` parameter.
- What's the best batch size to learn in this problem? `batch`
- What's the best learning rate? `lr`
- How many epochs does it take us to get complete the whole datast? `epochs`

## Loading dependencies

In [13]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

import torch
from torch.nn import Embedding, CrossEntropyLoss, Linear, ReLU
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.nn import radius_graph
from torch_geometric.utils import scatter
from sklearn.model_selection import train_test_split


import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
import pandas as pd
import json

## Data load

We load coordinates from nc files using the MDanalysis library.

We need to define each atom and residue index first. Tweak these files if necessary.

In [14]:
# u = mda.Universe("../data/md-2025-03-11/FP11ns.prmtop", "../data/md-2025-03-11/FP11-150ns.nc")
# residue_index = dict((key, id) for id, key in enumerate(np.unique(u.select_atoms('name CA').resnames).tolist()))
# atom_type_index = dict((key, id) for id, key in enumerate(np.unique(u.select_atoms('protein').atoms.types.tolist())))

with open('atom-list.json') as f: 
    atom_type_index = json.load(f)
with open('residue-list.json') as f: 
    residue_index = json.load(f)

The following function loads each frame in a `Data` instance:
- coordinates (`x`)
- residue indices (`res`)
- atom type indices (`types`)
- labels (`label`)

You can check how `Data` works in [https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html](https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html)

In [4]:
def trajectory_to_data(u, selection_string, residue_index, atom_type_index, label):
    for fr in u.trajectory:
        x = torch.from_numpy(u.select_atoms(selection_string).positions)
        residues = list(map(lambda x: residue_index[x], u.select_atoms(selection_string).resnames))
        atom_types = list(map(lambda x: atom_type_index[x], u.select_atoms(selection_string).atoms.types.tolist()))
        d = Data(pos=x, res=torch.tensor(residues, dtype=torch.long), types=torch.tensor(atom_types, dtype=torch.long), label=torch.tensor(label, dtype=torch.long))
        yield d

We load for all the files defined above

In [15]:

trj = []

for file in data_files:

    u = mda.Universe(file['topology_filename'], file['coordinates_filename'])
    trj += list(trajectory_to_data(u, selection_string, residue_index, atom_type_index, 0))

trj[:5]

[Data(pos=[746, 3], res=[746], types=[746], label=0),
 Data(pos=[746, 3], res=[746], types=[746], label=0),
 Data(pos=[746, 3], res=[746], types=[746], label=0),
 Data(pos=[746, 3], res=[746], types=[746], label=0),
 Data(pos=[746, 3], res=[746], types=[746], label=0)]

## Model 

Let's try to avoid tweaking it for now.

In [6]:
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.rg = radius_graph
        self.embed_res = Embedding(num_embeddings=21, embedding_dim=64)
        self.embed_type = Embedding(num_embeddings=50, embedding_dim=64)
        # 
        self.conv1_res = GCNConv(64, 32)
        self.relu1_res = ReLU()
        self.conv2_res = GCNConv(32, 16)
        # 
        self.conv1_type = GCNConv(64, 32)
        self.relu1_type = ReLU()
        self.conv2_type = GCNConv(32, 16)
        # 
        self.linear1 = Linear(32, 2)

    def forward(self, data):

        pos, res, type, batch = data.pos, data.res, data.batch, data.batch
        edge_index = radius_graph(pos, r=10.0, batch=batch)

        z_res = self.embed_type(res)
        z_res = self.conv1_res(z_res, edge_index)
        z_res = self.relu1_res(z_res)
        z_res = self.conv2_res(z_res, edge_index)

        z_type = self.embed_type(type)
        z_type = self.conv1_type(z_type, edge_index)
        z_type = self.relu1_type(z_type)
        z_type = self.conv2_type(z_type, edge_index)
        z_res = scatter(z_res, data.batch, dim=0, reduce='mean')
        z_type = scatter(z_type, data.batch, dim=0, reduce='mean')
        z = torch.cat([z_res, z_type], dim=1)
        z = self.linear1(z)
        return F.softmax(z, dim=1)

## Training / testing

In [7]:
train_trj, test_trj = train_test_split(trj, test_size=test_size, random_state=random_state)

In [8]:
gcn = GCN()
loss = CrossEntropyLoss()
optimizer = torch.optim.Adam(gcn.parameters(), lr=lr, weight_decay=5e-4)
train_loader = DataLoader(train_trj, batch_size=batch_size)
test_loader = DataLoader(test_trj, batch_size=batch_size)
history = []

for i in tqdm.tqdm(range(epochs)):
    statistics = dict()
    for d in train_loader:
        z = gcn(d)
        l = loss(z, d.label)
        l.backward()
        optimizer.step()
    statistics['training_loss'] = l.item()
    with torch.no_grad():
        statistics['test_accuracy'] = 0.0
        statistics['test_loss'] = 0.0
        for d in test_loader:
            z = gcn(d)
            l = loss(z, d.label)
            a = ((gcn(d).argmax(1) == d.label).sum())
            statistics['test_accuracy'] += a.item()
            statistics['test_loss'] += l.item()
    statistics['test_accuracy'] /= len(test_trj)
    statistics['test_loss'] /= ((len(test_trj) // batch_size) + 1)
    
    history.append(statistics)

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [02:06<00:00, 25.31s/it]


## Results

In [10]:
pd.DataFrame.from_records(history)

,training_loss,test_accuracy,test_loss
0,0.313595,1.0,0.309457
1,0.313262,1.0,0.309140
2,0.313262,1.0,0.309140
3,0.313262,1.0,0.309140
4,0.313262,1.0,0.309140


## Future steps

- More informative learning?
- Can we use a better scheme for train/test split than just random? (e.g. selecting the initial 5 ns)